In [ ]:
!pwd


In [ ]:
!nvidia-smi


# Create Environment


In [ ]:
!test -d /content/edge-lipsync-model || git clone https://github.com/monkira99/edge-lipsync-model.git /content/edge-lipsync-model


In [ ]:
%cd /content/edge-lipsync-model


In [ ]:
!git pull


In [ ]:
!uv sync


In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or environment before running Hub commands."


In [ ]:
!hf auth login --token "$HF_TOKEN" --quiet

!uv run python tools/hf_model_assets.py pull \
  --repo-id tiennguyenbnbk/edge-lipsync-model-assets \
  --local-dir models


In [ ]:
!ls -lah /content/edge-lipsync-model/models/mediapipe


# Build Hugging Face `datasets` Dataset


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/data")
DATASET_ROOT = DATA_DIR / "hdtf_xdub_duix_dataset"
WORK_DIR = DATA_DIR / "hdtf_xdub"
SOURCE_REPO = "Pinch-Research/lipsync-hdtf-training-data"
VIDEO_PREFIX = "xdub_teacher_pairs/videos"
SPEAKER_ID = "SaxbyChambliss"
OUTPUT_REPO = f"tiennguyenbnbk/hdtf-xdub-duix-dataset-{SPEAKER_ID}"
MAX_VIDEOS = 0  # 0 means all selected videos for the speaker.

WENET_ONNX = Path("/content/edge-lipsync-model/models/wenet/wenet.onnx")
FACE_LANDMARKER = Path("/content/edge-lipsync-model/models/mediapipe/face_landmarker.task")


In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"dataset_root={DATASET_ROOT}")
print(f"work_dir={WORK_DIR}")
print(f"output_repo={OUTPUT_REPO}")


In [ ]:
!ls -lah /content && ls -lah /content/data


In [ ]:
!uv run python tools/build_hf_video_dataset.py \
  --repo-id {SOURCE_REPO} \
  --dataset-root {DATASET_ROOT} \
  --work-dir {WORK_DIR} \
  --wenet-onnx {WENET_ONNX} \
  --landmark-model-asset-path {FACE_LANDMARKER} \
  --video-prefix {VIDEO_PREFIX} \
  --max-videos 0 \
  --list-speakers


In [ ]:
!uv run python tools/build_hf_video_dataset.py \
  --repo-id {SOURCE_REPO} \
  --dataset-root {DATASET_ROOT} \
  --work-dir {WORK_DIR} \
  --wenet-onnx {WENET_ONNX} \
  --landmark-model-asset-path {FACE_LANDMARKER} \
  --video-prefix {VIDEO_PREFIX} \
  --max-videos {MAX_VIDEOS} \
  --speaker-id {SPEAKER_ID} \
  --dry-run


In [ ]:
!uv run python tools/build_hf_video_dataset.py \
  --repo-id {SOURCE_REPO} \
  --dataset-root {DATASET_ROOT} \
  --work-dir {WORK_DIR} \
  --wenet-onnx {WENET_ONNX} \
  --landmark-model-asset-path {FACE_LANDMARKER} \
  --video-prefix {VIDEO_PREFIX} \
  --max-videos {MAX_VIDEOS} \
  --speaker-id {SPEAKER_ID} \
  --push \
  --hf-output-repo-id {OUTPUT_REPO} \
  --strict

!uv run python tools/hf_dataset.py pull \
  --repo-id {OUTPUT_REPO} \
  --cache-dir /content/data/hf_cache
